# Multi-Domain Voice Parkinson's Screening Benchmark
This notebook trains a specialized Tri-Modal model for `Voice_Dataset` and trains an **Acoustic Domain Router** to preserve the **97.44% Accuracy on MDVR-KCL** while delivering high performance on the new acoustic domain.

In [ ]:
# Step 1: Check GPU acceleration
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive (Optional but recommended for storing weights)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Install system tools and Python requirements
!pip install -r requirements.txt
!apt-get install -y ffmpeg

In [ ]:
# Step 4: Prepare MDVR-KCL Dataset
!mkdir -p data/raw/mdvr_kcl
!wget -O data/raw/mdvr_kcl/26_29_09_2017_KCL.zip "https://zenodo.org/records/2867216/files/26_29_09_2017_KCL.zip?download=1"
!unzip -q data/raw/mdvr_kcl/26_29_09_2017_KCL.zip -d data/raw/mdvr_kcl/

In [ ]:
# Step 5: Extract Voice_Dataset (Upload voice_dataset.zip or copy from Google Drive)
!mkdir -p data/raw/voice_dataset
# If uploaded to root:
!unzip -q voice_dataset.zip -d data/raw/voice_dataset/
# Check loaded file counts
!ls data/raw/voice_dataset/Healthy | wc -l
!ls data/raw/voice_dataset/Parkinsons | wc -l

In [ ]:
# Step 6: Extract KCL 97.44% Best Checkpoint (Upload parkinson_tri_modal_97_44_best.zip)
!unzip -q parkinson_tri_modal_97_44_best.zip -d /content/checkpoints/
!ls -la /content/checkpoints/

In [ ]:
# Step 7: Train Specialized Tri-Modal Ensemble on Voice_Dataset
!python run_experiments.py \
    --dataset voice_dataset \
    --data_dir ./data/raw/voice_dataset \
    --experiment tri_modal \
    --eval_strategy split \
    --save_model_path /content/checkpoints/parkinson_tri_modal_voice_dataset_best.joblib

In [ ]:
# Step 8: Train the Acoustic Domain Router
# This routes incoming audio based on acoustic characteristics (KCL vs Voice_Dataset)
!python train_domain_router.py \
    --kcl_features ./data/processed/mdvr/features_acoustic_gtcc.npz \
    --voice_features ./data/processed/voice_dataset/features_acoustic_gtcc.npz \
    --kcl_model /content/checkpoints/parkinson_tri_modal_97_44_best.joblib \
    --voice_model /content/checkpoints/parkinson_tri_modal_voice_dataset_best.joblib \
    --output_router_path /content/checkpoints/domain_router_best.joblib

In [ ]:
# Step 9: Re-verify MDVR-KCL Accuracy to confirm 100% preservation (97.44% Accuracy)
!python run_experiments.py \
    --dataset mdvr \
    --data_dir ./data/raw/mdvr_kcl \
    --experiment tri_modal \
    --eval_strategy split \
    --task READ_TEXT \
    --decision_threshold 0.56